# Did IBM's quantum hardware beat random guessing on its own benchmark?

This notebook answers that in about ten seconds using **only files from IBM's own published data
repository**. Nothing is downloaded to your machine, nothing is installed, and no data from the
audit repository is used. Every number below is computed live from
[github.com/jrm874/sqd_data_repository](https://github.com/jrm874/sqd_data_repository), the archive
named in the data-availability statement of
[Robledo-Moreno et al., *Sci. Adv.* **11**, eadu9991 (2025)](https://doi.org/10.1126/sciadv.adu9991).

**The comparison.** That archive ships, side by side, the energies their SQD pipeline reached from
(a) real quantum-hardware samples and (b) **uniform random bitstrings**, their own null control, on
the [2Fe-2S] system. Both run through the same pipeline at the same subspace dimensions. The
archive therefore already contains the experiment that asks whether the quantum samples did any work.

**What is claimed, and what is not.** This compares energies from their released files at matched
subspace dimension. It is not a claim of misconduct, and it does not dispute that SQD yields a valid
variational upper bound. Lower is better; every value here sits above their own DMRG reference of
-116.6056091 Ha.

Run the cell. Method follows [`_ibmuniform2.py`](https://github.com/PureStateLabs/sqd-spin-referee/blob/main/_ibmuniform2.py); paper section 2.7.

In [ ]:
import io, urllib.request, numpy as np
from scipy import stats

IBM = ("https://raw.githubusercontent.com/jrm874/sqd_data_repository/main/"
       "experiments/2Fe-2S")
E2 = -116.6056091          # their own DMRG reference, from their archive

def fetch(rel):
    with urllib.request.urlopen(f"{IBM}/{rel}", timeout=60) as r:
        raw = r.read()
    a = np.loadtxt(io.StringIO(raw.decode()), skiprows=2)
    return a.reshape(-1, a.shape[-1]), raw

hw, un, rawun = {}, {}, {}
for t in "ABC":
    hw[t], _ = fetch(f"sqd_hardware_energetics/energy-variance_data_SQD_eigenstate_{t}.txt")
    un[t], rawun[t] = fetch(f"sqd_on_uniform_distribution/"
                            f"energy-variance_data_SQD_eigenstate_{t}_uniform.txt")
print("fetched 6 files from IBM's repository\n")

# ------------------------------------------------------------------ structure
# The three published eigenstate files are NOT three independent experiments.
# Verify the real structure before counting anything.
rows = lambda r: set(map(tuple, r))
b_in_c  = rows(hw["B"]) <= rows(hw["C"])
un_same = rawun["B"] == rawun["C"]
a_disj  = not (rows(hw["A"]) & rows(hw["C"]))
print("EVIDENCE STRUCTURE (verified, not assumed)")
print(f"  hardware B is a strict subset of C : {b_in_c}  (C adds "
      f"{len(rows(hw['C']) - rows(hw['B']))} rows)")
print(f"  uniform B and C byte-identical     : {un_same}")
print(f"  hardware A disjoint from B and C   : {a_disj}")
assert b_in_c and un_same and a_disj
print("  => TWO independent streams: A, and B/C counted once (not three)\n")

def welch(a, b):
    ma, mb = a.mean(), b.mean(); va, vb = a.var(ddof=1), b.var(ddof=1)
    na, nb = len(a), len(b); se = np.sqrt(va / na + vb / nb)
    df = se**4 / ((va / na)**2 / (na - 1) + (vb / nb)**2 / (nb - 1))
    t = stats.t.ppf(0.975, df)
    return ma - mb, (ma - mb) - t * se, (ma - mb) + t * se

n = n_un = n_sig = 0
adv = {"A": [], "B/C": []}
sds = []
print(f"{'stream':7s}{'subspace dim':>14s}{'hardware':>11s}{'uniform':>10s}"
      f"{'gap':>8s}{'95% CI':>17s}{'p':>9s}")
print(f"{'':7s}{'':14s}{'mHa above reference':>21s}{'':8s}{'(mHa)':>17s}")
print("-" * 76)
for tag, H, U in [("A", hw["A"], un["A"]), ("B/C", hw["C"], un["B"])]:
    shared = sorted(set(H[:, 2].astype(np.int64)) & set(U[:, 2].astype(np.int64)))
    for d in shared:
        eh = H[H[:, 2].astype(np.int64) == d, 1]
        eu = U[U[:, 2].astype(np.int64) == d, 1]
        if len(eh) < 2 or len(eu) < 2:
            continue
        n += 1
        dm, lo, hi = welch(eh, eu)
        p = stats.mannwhitneyu(eu, eh, alternative="two-sided").pvalue
        n_un += eu.mean() <= eh.mean()
        n_sig += p < 0.05
        adv[tag].append(dm * 1e3)
        sds += [eh.std(ddof=1) * 1e3, eu.std(ddof=1) * 1e3]
        print(f"{tag:7s}{d:14,d}{(eh.mean()-E2)*1e3:11.1f}{(eu.mean()-E2)*1e3:10.1f}"
              f"{dm*1e3:8.1f}  [{lo*1e3:6.1f},{hi*1e3:6.1f}]{p:9.4f}")
print("-" * 76)
print("\nRESULT")
print(f"  uniform random beat the hardware (mean energy) : {n_un} of {n} conditions")
print(f"  of those, significant at p < 0.05 (two-sided)  : {n_sig} of {n}")
print(f"  margin, stream A   : {min(adv['A']):.1f} to {max(adv['A']):.1f} mHa")
print(f"  margin, stream B/C : {min(adv['B/C']):.1f} to {max(adv['B/C']):.1f} mHa")
print(f"  per-batch scatter  : {min(sds):.1f} to {max(sds):.1f} mHa "
      f"(the margin is not batch noise)")
for tag, H, U in [("A", hw["A"], un["A"]), ("B/C", hw["C"], un["B"])]:
    print(f"  best single energy, hardware minus uniform, {tag:4s}: "
          f"{(H[:,1].min()-U[:,1].min())*1e3:+.2f} mHa")
print("\n  Positive numbers mean uniform random did BETTER.")

## Reading the output

`gap` is hardware mean minus uniform mean, in millihartree. **Positive means the uniform random
control landed lower, that is, better.** The uniform control is ahead at every matched condition,
in both streams, by margins several times the per-batch scatter.

Two caveats, stated as the paper states them:

- Dimensions **within** a stream re-process the same released sample pool, so the nine conditions
  are repeated measures on two independent evidence streams, not nine independent experiments. No
  global significance level is attached across them; the per-condition tests stand on their own.
- This is [2Fe-2S], the flagship system small enough for uniform sampling to compete. On [4Fe-4S]
  at 36 orbitals hardware does beat uniform, as it must, since random 72-bit strings essentially
  never land in the physical sector. The [4Fe-4S] problem is different and worse: their own
  classical HCI file in the same archive beats their own quantum result there by about 149 mHa.

## If this reproduces, one thing follows

Whatever accuracy the flagship [2Fe-2S] numbers carry, the quantum samples are not where it comes
from. The archive's own recovery-ablation study points the same way: with configuration recovery
switched off, their noiseless N2 errors are **1.1 to 1.8 Hartree**; switched on, **4 to 7 mHa**.
Three orders of magnitude, at zero noise, from classical post-processing.

## Got something different?

Please say so: [open an issue](https://github.com/PureStateLabs/sqd-spin-referee/issues/new?template=reproduction_report.md)
or add a row to [RESULTS.md](https://github.com/PureStateLabs/sqd-spin-referee/blob/main/RESULTS.md).
Disagreements are more useful than confirmations.

Full context: [the paper](https://doi.org/10.5281/zenodo.21359923) (section 2.7), the
[plain-English writeup](https://purestatelabs.github.io/sqd-spin-referee/), and the
[short version](https://github.com/PureStateLabs/sqd-spin-referee/blob/main/SUMMARY.md).